In [1]:
#Setup
%pip -q install plotly pandas scipy numpy sympy anywidget
import import_setup
from gu_toolkit import *
from Fourier_Sounds_helper import *


Note: you may need to restart the kernel to use updated packages.


# Some functions and how they sound
A pure sine wave has a single frequency, so its graph is smooth and its sound is clear and simple. Triangle and truare waves sound very different because they are built from many sine waves mixed together. In other words, their shapes already hint that they contain richer harmonic content.

We will use the two periodic functions, the *square wave* and the *triangle wave*

$$
\mathrm{Sq}(x):=
\begin{cases}
1, & 0<x<\tfrac12,\\[4pt]
-1, & \tfrac12<x<1,
\end{cases}
$$
 and
$$
\mathrm{Tr}(x):=
\begin{cases}
4x, & 0\le x\le \tfrac14,\\[4pt]
2-4x, & \tfrac14\le x\le \tfrac34,\\[4pt]
4x-4, & \tfrac34\le x\le 1,
\end{cases}
$$
both extended periodically with period $1$.


Let us plot a sine wave, a square wave, and a triangle wave with adjustable amplitude and frequency, and then listen to how their shapes affect the sound.


In [2]:
# Plot: Comparing three wave forms
fig = Figure(x_range=(0, 1/220*4), y_range=(-2, 2), samples=1200)
fig.show()

with fig:
    set_title("Comparing three wave forms")
    plot(a * sin(2*pi*b * x), x, label=r"$a\sin(2 \pi b x)$", id="sin")
    plot(a * Tr(b * x), x, label=r"$a\,\mathrm{Tr}(b x)$", id="Tr")
    plot(a * Sq(b * x), x, label=r"$a\,\mathrm{Sq}(b x)$", id="Sq" )

    parameter(a,min=0.0, max=1.0, value=0.7, step=0.1)
    parameter(b, min=50, max=880.0, value=220, step=0.1)

OneShotOutput()

Let us add a mystery function to our plot. It is built as a sum of five sine waves with different amplitudes:

$$
F(x)=\sum_{n=1}^{5} c_n\sin(2\pi n x).
$$

The coefficients $c_n$ are hidden from us, so we only see and hear the final waveform. Our goal is to recover those coefficients from the graph.


In [3]:
F=create_mystery_function(5)
with fig:
    plot(a * F(b * x), x, label=r"$a\,F(bx)$", id="F")


# Finding the coefficients manually

How do we find the coefficients $a_1,\dots,a_5$? We can first adjust them by hand to see how each sine term changes the waveform, and afterwards compute the correct values directly from Fourier integrals.

Let us define our model sum.

In [4]:
F_model=sum (a[j] * sin(2*pi * j *x) for j in range(6))
display(Latex("Model:"))
display(F_model)

<IPython.core.display.Latex object>

a_1*sin(2*pi*x) + a_2*sin(4*pi*x) + a_3*sin(6*pi*x) + a_4*sin(8*pi*x) + a_5*sin(10*pi*x)

In [5]:
# Plot: Fitting the model
fig = Figure(x_range=(0, 1/220), y_range=(-2, 2), samples=1200)
fig.show()

with fig:
    set_title("Fitting the mystery function with five sine modes")
    plot(F(220 * x), x, label=r"$F(220 x)$", id="F")
    fig.plot(F_model.subs(x, 220*x), x, label=r"$\mathrm{model}$", id="model",autonormalization=True)
    for j in range(1, 6):
        plot(a[j] * sin(2*pi * j * 220 * x), x,
             label=fr"$a_{{{j}}}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.15, autonormalization=True)
        parameter(a[j], min=-1.2, max=1.2, value=0.0, step=0.05)


OneShotOutput()

# Finding the Fourier Coefficients

In [6]:
fig_interaction = Figure(x_range=(0, 1), y_range=(-3, 3))
fig_interaction.show()
with fig_interaction:
    set_title(r"Products of trig functions")
    plot(sin(2*pi*b[1]*x), x, label=r"$\sin(2 \pi b_1 x)$",id="sin_a1",alpha=0.3)
    plot(sin(2*pi*b[2]*x), x, label=r"$\sin(2 \pi b_2 x)$",id="sin_a2", alpha=0.3)
    plot(sin(2*pi*b[1]*x)*sin(2*pi*b[2]*x), x, label="product", id="product",  width=5)
    parameter(b[1], value=0, min=0, max=20, step=1)
    parameter(b[2], value=0, min=0, max=20, step=1)

OneShotOutput()

## Orthogonality of trigonometric functions

We have shown the following orthogonality relations on the interval $[0,1]$:

$$
\int_0^1 \sin(2\pi m x)\sin(2\pi n x)\,dx=
\begin{cases}
0,& m\ne n,\\[4pt]
\tfrac12,& m=n\ge 1,
\end{cases}
$$
$$
\int_0^1 \cos(2\pi m x)\cos(2\pi n x)\,dx=
\begin{cases}
0,& m\ne n,\\[4pt]
\tfrac12,& m=n\ge 1,
\end{cases}
$$
$$
\int_0^1 \sin(2\pi m x)\cos(2\pi n x)\,dx=0.
$$
Therefore, if

$$
F(x)=a_0+\sum_{n=1}^{\infty} a_n \cos(2\pi n x)+b_n \sin(2\pi n x),
$$

then multiplying by $\cos(2\pi n x)$ and integrating gives

$$
a_n=2\int_0^1 F(x)\cos(2\pi n x)\,dx,
$$

and multiplying by $\sin(2\pi n x)$ and integrating gives

$$
b_n=2\int_0^1 F(x)\sin(2\pi n x)\,dx.
$$


**Let us check this in practice by computing the coefficients of the mystery function with explicit numerical integrals.**


In [7]:
def fourier_coefficients(expr, Nmax, interval=(0, 1)):
    left, right = interval
    cos_coeffs = [2 * NIntegrate(expr * cos(2*pi*n*x), (x, left, right)) for n in range(0, Nmax + 1)]
    sin_coeffs = [2 * NIntegrate(expr * sin(2*pi*n*x), (x, left, right)) for n in range(0, Nmax + 1)]
    return cos_coeffs, sin_coeffs

In [8]:
# Compute Fourier coefficients of the mystery function using explicit numerical integration.
Nmax = 5
an_F, bn_F = fourier_coefficients(F.expr, Nmax, interval=(0, 1))

## Display coefficients
display(Markdown("Fourier coefficients for the mystery function $F$:"))

lines = [
    f"&a_{{{n}}} = {an_F[n]:.6f}  & &b_{{{n}}} = {bn_F[n]:.6f}"
    for n in range(0, Nmax + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(lines)}
\\end{{aligned}}
$$
"""))

## Model sum
model_F = an_F[0] + sum(
    an_F[n] * cos(2*pi*n*x) + bn_F[n] * sin(2*pi*n*x)
    for n in range(1, Nmax + 1)
)

display(Markdown(f"""Recovered Fourier model:
$$
{latex(model_F)}
$$
"""
))


Fourier coefficients for the mystery function $F$:

 
$$
\begin{aligned}
&a_{0} = -0.000000  & &b_{0} = 0.000000\\
&a_{1} = -0.000000  & &b_{1} = 0.315544\\
&a_{2} = -0.000000  & &b_{2} = 0.210363\\
&a_{3} = -0.000000  & &b_{3} = 0.420725\\
&a_{4} = -0.000000  & &b_{4} = 0.262953\\
&a_{5} = -0.000000  & &b_{5} = -0.368135
\end{aligned}
$$


Recovered Fourier model:
$$
0.315543949716871 \sin{\left(2 \pi x \right)} + 0.210362633144581 \sin{\left(4 \pi x \right)} + 0.420725266289161 \sin{\left(6 \pi x \right)} + 0.262953291430726 \sin{\left(8 \pi x \right)} - 0.368134608003016 \sin{\left(10 \pi x \right)} - 1.16853907136521 \cdot 10^{-16} \cos{\left(2 \pi x \right)} - 2.59709957550334 \cdot 10^{-17} \cos{\left(4 \pi x \right)} - 1.32744624919155 \cdot 10^{-16} \cos{\left(6 \pi x \right)} - 4.84200582519616 \cdot 10^{-17} \cos{\left(8 \pi x \right)} - 4.80663754084106 \cdot 10^{-17} \cos{\left(10 \pi x \right)} - 1.28655792500452 \cdot 10^{-16}
$$


In [9]:
# Plot: Fitting the model again, now with the coefficients recovered from integrals.
fig = Figure(x_range=(0, 1/220), y_range=(-2, 2), samples=1200)
fig.show()

dynamic_model=a[0] + sum(
    a[n] * cos(2*pi*n*x) + b[n] * sin(2*pi*n*x)
    for n in range(1, Nmax + 1)
)

with fig:
    set_title("Recovered Fourier model for the mystery function")
    plot(F(220 * x), x, label=r"$F(220 x)$", id="F", width=2)
    plot(dynamic_model.subs(x, 220*x), x, label=r"$\mathrm{model}$", id="model", width=1, autonormalization=True)
    for j in range(1, Nmax+1):
        plot(bn_F[j] * sin(2*pi * j * 220 * x), x,
             label=fr"${bn_F[j]:.3f}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.3)


OneShotOutput()

## What about the other waves?
The same method works for any periodic waveform. truare and triangle waves are especially interesting because their Fourier coefficients follow clear patterns: the truare wave uses only odd sine modes with slowly decaying coefficients, while the triangle wave also uses only odd sine modes but with much faster decay. Let us compute a few coefficients numerically and compare the resulting partial sums to the original waves.


In [10]:
Nmax=100

an_Sq, bn_Sq = fourier_coefficients(Sq(x), Nmax)


lines = [
    f"&a_{{{n}}} = {an_Sq[n-1]:.6f}  & &b_{{{n}}} = {bn_Sq[n-1]:.6f}"
    for n in range(0, 10 + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(lines)}
\\end{{aligned}}
$$
"""))

## TODO: introduce MD as alias to display(Markdown()), but if provided a sequence be smart about it. If strings then display as concatenated markdown (with one newline separating the strings)
## If there are rich objects then try to display them e.g. sympy -> latex (inline), Dataframe -> block display table (use display())

## The partial sum, n from 0 to N
N=symbols("N")
SN_Sq=an_Sq[0]+ sum( [
        Piecewise((1,n<=N),(0,True)) * an_Sq[n]*sin(2*pi*n*x)
        +
        Piecewise((1,n<=N),(0,True)) * bn_Sq[n]*sin(2*pi*n*x)
    for n in range(1,Nmax+1)
    ])    

#TODO Introduce the functions One(condition) that is Piecewise((1,condition),(0,True))

C:\Users\guraltsev\Documents\notebooks\gu_toolkit\src\gu_toolkit\numeric_operations.py:162: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  value, _error = quad(_integrand, _to_quad_limit(a), _to_quad_limit(b))


 
$$
\begin{aligned}
&a_{0} = 0.000000  & &b_{0} = 0.000000\\
&a_{1} = 0.000000  & &b_{1} = 0.000000\\
&a_{2} = 0.000000  & &b_{2} = 1.273240\\
&a_{3} = -0.000000  & &b_{3} = 0.000000\\
&a_{4} = 0.000000  & &b_{4} = 0.424413\\
&a_{5} = 0.000000  & &b_{5} = -0.000000\\
&a_{6} = 0.000000  & &b_{6} = 0.254648\\
&a_{7} = -0.000000  & &b_{7} = 0.000000\\
&a_{8} = 0.000000  & &b_{8} = 0.181891\\
&a_{9} = 0.000000  & &b_{9} = -0.000000\\
&a_{10} = -0.000000  & &b_{10} = 0.141471
\end{aligned}
$$


In [11]:
fig_Sq = Figure(x_range=(0, 1/220), y_range=(-1.5, 1.5), samples=1600)
fig_Sq.show()
with fig_Sq:
    set_title("Square wave versus their sine approximations")
    plot(Sq(220*x), x, label=r"$\mathrm{Sq}(x)$", id="tr")
    plot(SN_Sq.subs(x,220*x), x, 
         label=r"$a_0+\sum_{n=0}^N a_n \cos(2π n x)+b_n \sin(2π n x)$", id="SN_Sq", 
         alpha=0.8, autonormalization=True)
    for j in range(1, 10):
        plot( Piecewise((1,j<=N),(0,True)) * bn_Sq[j] * sin(2*pi * j * 220 * x), x,
             label=fr"${bn_Sq[j]:.3f}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.3)
         
    parameter(N,value=0,min=0,max=Nmax,step=1)

## TODO: make legend be vertically bounded and display a vertical scroll 


OneShotOutput()

## Same for the triangle wave

In [12]:
Nmax=100

an_Tr, bn_Tr = fourier_coefficients(Tr(x), Nmax)


lines = [
    f"&a_{{{n}}} = {an_Tr[n-1]:.6f}  & &b_{{{n}}} = {bn_Tr[n-1]:.6f}"
    for n in range(0, 10 + 1)
]

display(Markdown(f""" 
$$
\\begin{{aligned}}
{"\\\\\n".join(lines)}
\\end{{aligned}}
$$
"""))

## The partial sum, n from 0 to N
N=symbols("N")
SN_Tr=an_Tr[0]+ sum( [
        Piecewise((1,n<=N),(0,True)) * an_Tr[n]*sin(2*pi*n*x)
        +
        Piecewise((1,n<=N),(0,True)) * bn_Tr[n]*sin(2*pi*n*x)
    for n in range(1,Nmax+1)
    ])    

C:\Users\guraltsev\Documents\notebooks\gu_toolkit\src\gu_toolkit\numeric_operations.py:162: IntegrationWarning: The maximum number of subdivisions (50) has been achieved.
  If increasing the limit yields no improvement it is advised to analyze 
  the integrand in order to determine the difficulties.  If the position of a 
  local difficulty can be determined (singularity, discontinuity) one will 
  probably gain from splitting up the interval and calling the integrator 
  on the subranges.  Perhaps a special-purpose integrator should be used.
  value, _error = quad(_integrand, _to_quad_limit(a), _to_quad_limit(b))


 
$$
\begin{aligned}
&a_{0} = 0.000000  & &b_{0} = 0.000000\\
&a_{1} = -0.000000  & &b_{1} = 0.000000\\
&a_{2} = -0.000000  & &b_{2} = 0.810569\\
&a_{3} = 0.000000  & &b_{3} = 0.000000\\
&a_{4} = -0.000000  & &b_{4} = -0.090063\\
&a_{5} = -0.000000  & &b_{5} = -0.000000\\
&a_{6} = 0.000000  & &b_{6} = 0.032423\\
&a_{7} = -0.000000  & &b_{7} = 0.000000\\
&a_{8} = 0.000000  & &b_{8} = -0.016542\\
&a_{9} = 0.000000  & &b_{9} = -0.000000\\
&a_{10} = 0.000000  & &b_{10} = 0.010007
\end{aligned}
$$


In [13]:
fig_Tr = Figure(x_range=(0, 1/220), y_range=(-1.5, 1.5), samples=1600)
fig_Tr.show()
with fig_Tr:
    set_title("Triangle wave versus their sine approximations")
    plot(Tr(220*x), x, label=r"$\mathrm{Tr}(x)$", id="tr", width=3, color="blue")
    plot(SN_Tr.subs(x,220*x), x, label=r"$a_0+\sum_{n=0}^N a_n \cos(2π n x)+b_n \sin(2π n x)$", id="SN_Tr", 
         alpha=0.8, width=2, color="red",autonormalization=True)
    for j in range(1, 10):
        plot( Piecewise((1,j<=N),(0,True)) * bn_Tr[j] * sin(2*pi * j * 220 * x), x,
             label=fr"${bn_Tr[j]:.3f}\sin(2\pi \cdot {j} \cdot 220 x)$",
             color="green", alpha=0.3)
    parameter(N,value=0,min=0,max=Nmax,step=1)

OneShotOutput()

# Interference

In [14]:
# Plot: Comparing three wave forms
fig = Figure(x_range=(0, 1/220*4), y_range=(-2, 2), samples=1200)
fig.show()

with fig:
    set_title("Comparing three wave forms")
    plot(a[1] * sin(2*pi*b[1] * x), x, label=r"$a_1\sin(2 \pi b_1 x)$", id="sin1")
    plot(a[2] * sin(2*pi*b[2] * x), x, label=r"$a_2\sin(2 \pi b_2 x)$", id="sin2")
    plot(a[1] * sin(2*pi*b[1] * x)+a[2] * sin(2*pi*b[2] * x), x, label=r"sum", id="sin-sum",   width=3 , autonormalization=True
)
    
    parameter(a[2],min=0.0, max=1.0, value=0.7, step=0.1)
    parameter(b[2], min=50, max=880.0, value=220, step=0.1)
    parameter(a[1],min=0.0, max=1.0, value=0.7, step=0.1)
    parameter(b[1], min=50, max=880.0, value=220, step=0.1)

OneShotOutput()

# Wave equation

In [15]:
fig_wave_initial_data=Figure(x_range=(-1/2,1/2), y_range=(-2,10))
fig_wave_initial_data.show()

u0=Piecewise((100*(x+1/10), (-1/10 < x) & (x < 0)), (-100*(x-1/10), (0<x) & (x<1/10)), (0, True))
v0=Piecewise((-10, (-1/10 < x) & (x < 0)),(10, (0<x) & (x<1/10)), (0, True))

with fig_wave_initial_data:
    plot(u0,x, label=r"$u_0$", id="u_0")
    plot(v0,x, label=r"$v_0$", id="v_0")



OneShotOutput()

In [16]:
A_FT_u0, B_FT_u0=NReal_Fourier_Series(u0, (x,-1/2,1/2))
A_FT_v0, B_FT_v0=NReal_Fourier_Series(v0, (x,-1/2,1/2))


display(Markdown("**Coefficients of initial value** $u_0$ **and initial velocity** $v_0$"))
display( pd.DataFrame({
    "$u_0$ $\\cos$ coeffs": A_FT_u0,
             "$u_0$ $\\sin$ coeffs": B_FT_u0,
   "$v_0$ $\\cos$ coeffs": A_FT_v0,
             "$v_0$ $\\sin$ coeffs": B_FT_v0
             }))








**Coefficients of initial value** $u_0$ **and initial velocity** $v_0$

,$u_0$ $\cos$ coeffs,$u_0$ $\sin$ coeffs,$v_0$ $\cos$ coeffs,$v_0$ $\sin$ coeffs
0,0.997500,0.000000e+00,0.000000e+00,0.000000
1,1.364761,2.096845e-16,2.430615e-17,0.861804
2,1.234101,3.618525e-16,-4.635535e-16,1.558619
3,1.038515,3.520066e-16,-5.944893e-16,1.967574
4,0.806510,3.377555e-16,-9.551577e-16,2.037931
...,...,...,...,...
1996,-0.003528,-5.356889e-16,2.987749e-16,-0.002058
1997,-0.003530,6.139459e-16,-6.195819e-16,-0.003352
1998,-0.003532,-1.430926e-15,1.321556e-15,-0.003359
1999,-0.003535,-2.501501e-16,1.317030e-16,-0.002078


In [17]:
Nmax=200
soln= [
(
(A_FT_u0[n]*cos(2*pi*n*x)+B_FT_u0[n]*sin(2*pi*n*x))*cos(2*pi*n*t)
    +
   1/n*(A_FT_v0[n]*cos(2*pi*n*x)+B_FT_v0[n]*sin(2*pi*n*x))*sin(2*pi*n*t) 
if n>0    
else
    A_FT_u0[n]/2+A_FT_v0[n]*t/2
)
    for n in range(0, Nmax+1)
]

In [18]:
fig_wave_soln=Figure(x_range=(-1/2,1/2), y_range=(-2,10))
fig_wave_soln.show()

n=1
display(soln[n])

solF= sum( [Piecewise((1,n<N),(0,True))*soln[n] for n in range(0,Nmax+1)])
with fig_wave_soln:
    plot(soln[n],x, label=r"$w_n$", id="w_n")
    plot(solF,x,label=r"$w(x,t)$", id="full_sol")
    parameter(t, min=0, max=10, value=0, step=0.01)
    parameter(N, min=0, max=Nmax, value=1, step=1)


OneShotOutput()

(2.09684464110707e-16*sin(2*pi*x) + 1.3647605056147*cos(2*pi*x))*cos(2*pi*t) + (0.861803538279689*sin(2*pi*x) + 2.43061518683397e-17*cos(2*pi*x))*sin(2*pi*t)

In [19]:
print(fig_wave_soln.performance_report(recent_event_limit=5))

Runtime diagnostics
  platform: win32
  is_pyodide: False
  timer_backend: asyncio_running_loop
  widget_support: {'anywidget_available': True, 'anywidget_is_fallback': False, 'figurewidget_supported': True, 'anywidget_mode': 'real_package', 'figurewidget_mode': 'FigureWidget', 'reason': None}
  schedule_later: {'state': {'last_backend': 'asyncio_running_loop', 'last_delay_ms': 10.000000009313226, 'last_owner': 'Figure.render'}, 'counters': {'backend_asyncio_running_loop': 17, 'schedule_calls': 17}}
  current_widget_type: FigureWidget

Runtime support scheduler
  state:
    last_backend: asyncio_running_loop
    last_delay_ms: 10.000000009313226
    last_owner: Figure.render
  counters:
    backend_asyncio_running_loop: 17
    schedule_calls: 17
  timings:
    requested_delay_ms: count=17 mean_ms=10.000 last_ms=10.000 max_ms=10.000
  recent_events:
    backend_asyncio_running_loop: +1
    requested_delay_ms: 10.000000009313226 ms
    schedule_calls: +1
    backend_asyncio_running_loop: